# Compare Ecological Metadata

This notebook compares extracted ecological metadata table.

Expected fields:

- `row_id`
- `site`
- `date/time`
- `end second`
- `start second`

The notebook is flexible with column names such as `date_time`, `datetime`, `end_second`, and `start_second`.

Outputs saved to `/kaggle/working/`:

- `ecological_metadata_comparison_summary.csv`
- `ecological_metadata_field_mismatches.csv`
- `ecological_metadata_only_in_dhyey.csv`
- `ecological_metadata_only_in_yunting.csv`

In [ ]:
# Change these paths based on where the files are in Kaggle
MY_FILE = "/kaggle/input/ecological-metadata/train_ecological_metadata.csv"
YUNTING_FILE = "/kaggle/input/YUNTING_DATASET_NAME/yunting_metadata.csv"

## Load both files

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

dhyey = pd.read_csv(MY_FILE)
yunting = pd.read_csv(YUNTING_FILE)

print("My file shape:", dhyey.shape)
print("Yunting file shape:", yunting.shape)

display(dhyey.head())
display(yunting.head())

## Standardize column names
This makes the comparison safer in case one file uses small formatting differences, such as:
`start_second` instead of `start second`

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace("_", " ", regex=False)
    )
    return df

dhyey = clean_column_names(dhyey)
yunting = clean_column_names(yunting)

print("\n\nDhyey columns:")
print(dhyey.columns.tolist())
print("------------------"*4)
print("\n\nYunting columns:")
print(yunting.columns.tolist())

## Keep only the columns needed for comparison

In [ ]:
required_cols = ["row id", "site", "date/time", "start second", "end second"]

# Rename row id back to row_id for easier merge
dhyey = dhyey.rename(columns={"row id": "row_id"})
yunting = yunting.rename(columns={"row id": "row_id"})

required_cols = ["row_id", "site", "date/time", "start second", "end second"]

missing_dhyey = [col for col in required_cols if col not in dhyey.columns]
missing_yunting = [col for col in required_cols if col not in yunting.columns]

print("Missing from Dhyey file:", missing_dhyey)
print("Missing from Yunting file:", missing_yunting)

if missing_dhyey or missing_yunting:
    raise ValueError("One of the files is missing required columns.")

dhyey_small = dhyey[required_cols].copy()
yunting_small = yunting[required_cols].copy()

display(dhyey_small.head())
display(yunting_small.head())

## Compare rows by row_id

In [ ]:
merged = dhyey_small.merge(
    yunting_small,
    on="row_id",
    how="outer",
    suffixes=("_dhyey", "_yunting"),
    indicator=True
)

print("Comparison summary:")
display(merged["_merge"].value_counts().reset_index().rename(
    columns={"index": "merge_status", "_merge": "count"}
))

only_in_dhyey = merged[merged["_merge"] == "left_only"].copy()
only_in_yunting = merged[merged["_merge"] == "right_only"].copy()
both = merged[merged["_merge"] == "both"].copy()

print("Rows only in Dhyey file:", len(only_in_dhyey))
print("Rows only in Yunting file:", len(only_in_yunting))
print("Rows in both files:", len(both))

## Check field-level matches
For rows that appear in both files, check whether site, date/time, start second, and end second match.

In [ ]:
compare_cols = ["site", "date/time", "start second", "end second"]

for col in compare_cols:
    both[f"{col} match"] = (
        both[f"{col}_dhyey"].astype(str).str.strip()
        == both[f"{col}_yunting"].astype(str).str.strip()
    )

match_cols = [f"{col} match" for col in compare_cols]

both["all fields match"] = both[match_cols].all(axis=1)

print("Rows where all fields match:", both["all fields match"].sum())
print("Rows with at least one mismatch:", (~both["all fields match"]).sum())

display(
    both["all fields match"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "all_fields_match", "all fields match": "count"})
)

## Show mismatched rows

In [ ]:
mismatches = both[~both["all fields match"]].copy()

cols_to_show = ["row_id"]

for col in compare_cols:
    cols_to_show.extend([f"{col}_dhyey", f"{col}_yunting", f"{col} match"])

display(mismatches[cols_to_show].head(50))

print("Total mismatched rows:", len(mismatches))

## Save comparison outputs
The notebook saves three files:
1. rows only in Dhyey file
2. rows only in Prof. Yin's file
3. rows where fields do not match

In [ ]:
only_in_dhyey.to_csv("/kaggle/working/metadata_only_in_dhyey.csv", index=False)
only_in_yunting.to_csv("/kaggle/working/metadata_only_in_yunting.csv", index=False)
mismatches.to_csv("/kaggle/working/metadata_field_mismatches.csv", index=False)

summary = pd.DataFrame({
    "metric": [
        "rows_in_my_file",
        "rows_in_yunting_file",
        "rows_only_in_my_file",
        "rows_only_in_yunting_file",
        "rows_in_both_files",
        "rows_with_all_fields_matching",
        "rows_with_field_mismatches",
    ],
    "value": [
        len(dhyey_small),
        len(yunting_small),
        len(only_in_dhyey),
        len(only_in_yunting),
        len(both),
        int(both["all fields match"].sum()),
        int((~both["all fields match"]).sum()),
    ]
})

summary.to_csv("/kaggle/working/metadata_comparison_summary.csv", index=False)

display(summary)

print("Saved:")
print("/kaggle/working/metadata_only_in_dhyey.csv")
print("/kaggle/working/metadata_only_in_yunting.csv")
print("/kaggle/working/metadata_field_mismatches.csv")
print("/kaggle/working/metadata_comparison_summary.csv")

## Interpretation

If everything matches, the important numbers should be:

- rows only in my file = 0
- rows only in Prof. Yin's file = 0
- rows with field mismatches = 0

If there are mismatches, check `metadata_field_mismatches.csv` to see which fields are different.